# Wasserstein Test

In this document i will test out a pipeline for using the Wasserstein distance measure to measure the distance between the two distributions of ngrams.

In [1]:
import sys

import pandas as pd

from from_root import from_root
from scipy.stats import wasserstein_distance

from common_ngrams import (
    find_common_ngrams_in_collections,
    find_occurrence_aware_common_ngrams,
    extract_ngrams,
    combine_ngrams,
    subtract_ngrams,
    filter_len_common_ngrams
)
from common_ngrams.tokenization import tokenize_to_tokens
from common_ngrams.preprocessing import preprocess_text

sys.path.insert(
    0,
    str(from_root("src")),
)

from read_and_write_docs import read_rds, read_jsonl, write_rds
from model_loading import load_model
from n_gram_scoring import score_ngrams_to_df
from utils import apply_temp_doc_id, build_metadata_df

/Users/user/Documents/GitHub/n-gram-av-methods/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer, model = load_model("/Volumes/BCross/models/gpt2")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 617.90it/s]


In [3]:
base_loc = '/Volumes/BCross/datasets/author_verification_updated/'
data_type = 'test'
corpus = 'Perverted Justice'
problem = 'iurutherford vs iurutherford'
scoring_model = 'gpt2'
max_context_tokens = 10

# Set the preprocessing variables
lowercase=True,
remove_punctuation=True,
remove_symbols=True,
remove_numeric=True,
remove_url=True,

# Add the locations
data_loc = f"{base_loc}/{data_type}/{corpus}"
metadata_loc = f"{base_loc}/{data_type}/metadata.rds"

# Load Data
known = read_jsonl(f"{data_loc}/known_raw.jsonl")
known = apply_temp_doc_id(known)

unknown = read_jsonl(f"{data_loc}/unknown_raw.jsonl")
unknown = apply_temp_doc_id(unknown)

metadata = read_rds(metadata_loc)
filtered_metadata = metadata[
    (metadata['corpus'] == corpus)
    & (metadata['problem'] == problem)
].copy()

# Preprocess data
agg_metadata = build_metadata_df(filtered_metadata, known, unknown)

known_author = filtered_metadata['known_author'].iloc[0]
unknown_author = filtered_metadata['unknown_author'].iloc[0]
    
selected_known = known[known['author'] == known_author]
selected_unknown = unknown[unknown['author'] == unknown_author]

unknown_doc = selected_unknown['doc_id'].iloc[0]
unknown_text = selected_unknown['text'].iloc[0]

unknown_text = preprocess_text(
    unknown_text,
    lowercase=lowercase,
    remove_punctuation=remove_punctuation,
    remove_symbols=remove_symbols,
    remove_numeric=remove_numeric,
    remove_url=remove_url
)

selected_known = selected_known.copy()

selected_known["text"] = selected_known["text"].apply(
    lambda text: preprocess_text(
        text,
        lowercase=lowercase,
        remove_punctuation=remove_punctuation,
        remove_symbols=remove_symbols,
        remove_numeric=remove_numeric,
        remove_url=remove_url,
    )
)

num_known_problems = len(selected_known)
print(f"There are {num_known_problems} known texts in the problem")

known_texts = selected_known['text'].tolist()

There are 2 known texts in the problem


In [4]:
common_ngrams = find_common_ngrams_in_collections(
    reference_texts = known_texts,
    comparison_texts = unknown_text,
    tokenizer=tokenizer,
    include_subgrams=True
)

oa_common_ngrams = find_occurrence_aware_common_ngrams(
    reference_texts = known_texts,
    comparison_texts = unknown_text,
    tokenizer=tokenizer
)

# Used for creating the lookup column
oa_ngram_set = {tuple(ngram) for ngram in oa_common_ngrams}

# Filter by token length if desired.
filtered_common_ngrams = filter_len_common_ngrams(
    common_ngrams,
    min_len=2,
)

distinct_lengths = sorted({len(ngram) for ngram in filtered_common_ngrams})

In [5]:
# Score the common ngrams
print("Scoring the Unknown n-grams")
unknown_scored_df = score_ngrams_to_df(
    filtered_common_ngrams[0:2],
    model,
    tokenizer,
    full_text=unknown_text,
    use_bos=True,
    num_tokens=max_context_tokens,
    greatest_common=False,
)

unknown_scored_df["occurrence_aware"] = unknown_scored_df["tokens"].apply(
    lambda tokens: tuple(tokens) in oa_ngram_set
)

unknown_metadata = {
    "data_type": data_type,
    "corpus": corpus,
    'scoring_model': scoring_model,
    'max_context_tokens': max_context_tokens,
    "problem": problem,
    "author": unknown_author,
    "texttype": "unknown",
    "doc_id": selected_unknown["doc_id"].iloc[0],
    "remove_punctuation": True
}

for col, value in reversed(list(unknown_metadata.items())):
    unknown_scored_df.insert(0, col, value)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Scoring the Unknown n-grams


KeyboardInterrupt: 

In [185]:
# Score known n-grams across all known texts
known_scored_dfs = []

for i, (_, row) in enumerate(selected_known.iterrows()):

    print(f"Scoring known text {i + 1}/{len(selected_known)}")

    df = score_ngrams_to_df(
        filtered_common_ngrams[0:2],
        model,
        tokenizer,
        full_text=row["text"],
        use_bos=True,
        num_tokens=max_context_tokens,
        greatest_common=False,
    )

    # Add occurrence-aware indicator
    df["occurrence_aware"] = df["tokens"].apply(
        lambda tokens: tuple(tokens) in oa_ngram_set
    )

    # Add metadata
    known_metadata = {
        "data_type": data_type,
        "corpus": corpus,
        'scoring_model': scoring_model,
        'max_context_tokens': max_context_tokens,
        "problem": problem,
        "author": row["author"],
        "texttype": "known",
        "doc_id": row["doc_id"],
        "remove_punctuation": True,
    }

    for col, value in reversed(list(known_metadata.items())):
        df.insert(0, col, value)

    known_scored_dfs.append(df)

# Combine known documents into a single DataFrame
known_scored_df = pd.concat(
    known_scored_dfs,
    ignore_index=True,
)

known_scored_df = known_scored_df.sort_values(
    by=["phrase_num", "doc_id", "phrase_occurrence"]
).reset_index(drop=True)

Scoring known text 1/2
Scoring known text 2/2


In [ ]:
no_context_df = score_ngrams_to_df(
    filtered_common_ngrams[0:2],
    model,
    tokenizer,
    full_text=None,
    use_bos=True
)

no_context_df["occurrence_aware"] = no_context_df["tokens"].apply(
    lambda tokens: tuple(tokens) in oa_ngram_set
)

no_context_metadata = {
    "data_type": data_type,
    "corpus": corpus,
    'scoring_model': scoring_model,
    'max_context_tokens': 0,
    "problem": problem,
    "author": None,
    "texttype": "no_context",
    "doc_id": None,
    "remove_punctuation": True
}

for col, value in reversed(list(no_context_metadata.items())):
    no_context_df.insert(0, col, value)

In [187]:
# Combine all scored df's into one to save
combined_scored_df = pd.concat(
    [unknown_scored_df, known_scored_df, no_context_df],
    ignore_index=True
)

In [6]:
# Ngram extraction for ngrams not in common of the same length
per_text_k = extract_ngrams(
    known_texts,
    tokenizer=tokenizer,
    n=distinct_lengths
)
whole_k = combine_ngrams(per_text_k)
remaining_k = subtract_ngrams(whole=whole_k, common=common_ngrams)
print(len(remaining_k))

per_text_u = extract_ngrams(
    unknown_text,
    tokenizer=tokenizer,
    n=distinct_lengths
)
whole_u = combine_ngrams(per_text_u)
remaining_u = subtract_ngrams(whole=whole_u, common=common_ngrams)
print(len(remaining_u))

remaining_ngrams = [
    list(ngram)
    for ngram in dict.fromkeys(
        tuple(ngram) for ngram in remaining_u + remaining_k
    )
]

print(f"Distinct remaining n-grams: {len(remaining_ngrams)}")

2560
1513
Distinct remaining n-grams: 4073


In [189]:
print("Scoring the Unknown n-grams")
unknown_scored_df_u = score_ngrams_to_df(
    remaining_u[0:2],
    model,
    tokenizer,
    full_text=unknown_text,
    use_bos=True,
    num_tokens=max_context_tokens,
    greatest_common=False,
)

unknown_scored_df_u["occurrence_aware"] = unknown_scored_df_u["tokens"].apply(
    lambda tokens: tuple(tokens) in oa_ngram_set
)

unknown_metadata = {
    "data_type": data_type,
    "corpus": corpus,
    'scoring_model': scoring_model,
    'max_context_tokens': max_context_tokens,
    "problem": problem,
    "author": unknown_author,
    "texttype": "unknown",
    "doc_id": selected_unknown["doc_id"].iloc[0],
    "remove_punctuation": True
}

for col, value in reversed(list(unknown_metadata.items())):
    unknown_scored_df_u.insert(0, col, value)

Scoring the Unknown n-grams


In [190]:
# Score known n-grams across all known texts
known_scored_dfs_k = []

for i, (_, row) in enumerate(selected_known.iterrows()):

    print(f"Scoring known text {i + 1}/{len(selected_known)}")

    df = score_ngrams_to_df(
        remaining_k[0:2],
        model,
        tokenizer,
        full_text=row["text"],
        use_bos=True,
        num_tokens=max_context_tokens,
        greatest_common=False,
    )

    # Add occurrence-aware indicator
    df["occurrence_aware"] = df["tokens"].apply(
        lambda tokens: tuple(tokens) in oa_ngram_set
    )

    # Add metadata
    known_metadata = {
        "data_type": data_type,
        "corpus": corpus,
        'scoring_model': scoring_model,
        'max_context_tokens': max_context_tokens,
        "problem": problem,
        "author": row["author"],
        "texttype": "known",
        "doc_id": row["doc_id"],
        "remove_punctuation": True,
    }

    for col, value in reversed(list(known_metadata.items())):
        df.insert(0, col, value)

    known_scored_dfs_k.append(df)

# Combine known documents into a single DataFrame
known_scored_df_k = pd.concat(
    known_scored_dfs_k,
    ignore_index=True,
)

known_scored_df_k = known_scored_df_k.sort_values(
    by=["phrase_num", "doc_id", "phrase_occurrence"]
).reset_index(drop=True)

Scoring known text 1/2
Scoring known text 2/2


In [ ]:
no_context_df_u = score_ngrams_to_df(
    remaining_ngrams[0:2],
    model,
    tokenizer,
    full_text=None,
    use_bos=True
)

no_context_df_u["occurrence_aware"] = no_context_df_u["tokens"].apply(
    lambda tokens: tuple(tokens) in oa_ngram_set
)

no_context_metadata = {
    "data_type": data_type,
    "corpus": corpus,
    'scoring_model': scoring_model,
    'max_context_tokens': 0,
    "problem": problem,
    "author": None,
    "texttype": "no_context",
    "doc_id": None,
    "remove_punctuation": True
}

for col, value in reversed(list(no_context_metadata.items())):
    no_context_df_u.insert(0, col, value)

In [192]:
# Combine all scored df's into one to save
combined_scored_df_u = pd.concat(
    [unknown_scored_df_u, known_scored_df_k, no_context_df_u],
    ignore_index=True
)